# The fresh p4 marketplace, validated

**The paid run is one command**, from the repository root, once the owner says go (it needs `pip install anthropic`
and `ANTHROPIC_API_KEY`):

```
python3 scripts/session/fresh_validation.py run --paid
```

Estimated cost, by step (fresh-marketplace.md 5.7; Opus 5.5 answers, Haiku 5.5 judges, two seeds):

| step | what is paid for | estimate |
|---|---|---|
| 1. tasks, 2. contexts | free: the short-turn and newer task sets, the migrated stores, a training round, the recall lines | $0.00 |
| 3. judge check | Haiku 5.5 grades every judged task's reference and bad answer (~200 tasks x 2) | $0.30 |
| 4. validation (8.2) | Opus 5.5 answers, 3 pathways x 2 seeds over 64 tasks; the built-in from the audit's cache, silent tasks free | $20.00 |
| 5. short turns (8.3) | Opus 5.5 answers ~150 turns x 2 pathways x 2 seeds, Haiku judge | $4.00 |
| 6. newer than the model (8.4) | Opus 5.5 answers 12 tasks x 2 pathways x 2 seeds, Haiku judge | $0.30 |
| 7. findings, 8. upgrades | free (the upgrades' paid re-measure is off: `UPGRADES_PAID = False`) | $0.00 |
| **total** | | **$24.60** (cap `TOTAL_MAX_USD` = $42) |

About $25 in all. No step starts once `TOTAL_MAX_USD` is spent, and the suite refuses any run whose estimate
is above what is left. Run the notebook itself instead with `FAKE = False` in the settings cell; with `FAKE = True`
(the default) every cell runs offline, with no key and no cost, as `python3 scripts/session/fresh_validation.py run`
does.

What it settles (fresh-marketplace.md 8.2-8.5):

| pathway | what it is |
|---|---|
| `builtin` | the built-in stand-in alone, answered from the audit's cache where the request is the same |
| `=recall@<store>` | plus the lines `p4.py query` prints for the task over the migrated store (none for most general tasks) |
| `=recall+pinned@<store>` | plus each store's `form:` and pinned rulings, at most 1,500 characters |
| `=emission-plain` | plus the 10-line plain rewrite of the emission gate's testable rules |

## Settings

In [ ]:
import os, sys, json, pathlib
REPO = next((p for p in [pathlib.Path(os.environ.get("P4_REPO", ".")).resolve(), *pathlib.Path.cwd().resolve().parents,
             pathlib.Path.cwd().resolve()] if (p / "scripts" / "session" / "fresh_validation.py").exists()), None)
sys.path.insert(0, str(REPO / "scripts" / "session"))
import fresh_validation as fv

FAKE = True                       # False: the paid run (the API key in ANTHROPIC_API_KEY)
TOTAL_MAX_USD = 42.0              # the whole run's cap: the design's ~$42, step 11's ~$25 and its reserve
ANSWER_MODEL = "claude-opus-5-5"
SEEDS = [0, 1]
JUDGE_MODELS = ['claude-haiku-5-5', 'claude-sonnet-5-5']   # the first that passes the grader check judges
WORK = os.environ.get("P4_FRESH_WORK", str(fv.WORK))
STORES = ""                       # NAME=PATH,...; "" is the four clones under P4_FIXTURES
GENERAL_REPO = "all"              # general tasks' recall from each store in turn, or one store
APPLY_UPGRADES = True             # commit the kept upgrades on this branch (never pushed)
UPGRADES_PAID = False             # re-measure the kept upgrades with the paid steps too
fv.configure(FAKE=FAKE, TOTAL_MAX_USD=TOTAL_MAX_USD, ANSWER_MODEL=ANSWER_MODEL, SEEDS=SEEDS, JUDGE_MODELS=JUDGE_MODELS,
             WORK=WORK, STORES=STORES, GENERAL_REPO=GENERAL_REPO, APPLY_UPGRADES=APPLY_UPGRADES,
             UPGRADES_PAID=UPGRADES_PAID)
print("FAKE" if FAKE else "PAID", "| cap $", TOTAL_MAX_USD, "| work", WORK)

## 1. Tasks (free)
The short turns held out of each migrated store (8.3) and the twelve newer-than-the-model tasks (8.4).

In [ ]:
tasks_info = fv.step_tasks()

## 2. Contexts (free)
One training round on each migrated store, then `p4.py query` for every task: the per-task context files of the recall pathways.

In [ ]:
manifest = fv.step_contexts()

## 3. The judge
Every judged task's reference must pass and every bad answer fail, on Haiku 5.5 first, Sonnet 5.5 if Haiku does not.

In [ ]:
if not FAKE:
    print('cache:', fv.seed_cache(), 'answers of the audit copied in')
JUDGE = fv.step_judge()

## 4. The first gate (8.2)

In [ ]:
outs = {'validate': fv.step_validate(JUDGE)}

## 5. Short turns, held out (8.3)

In [ ]:
outs['short'] = fv.step_short(JUDGE)

## 6. Newer than the model (8.4)

In [ ]:
outs['newer'] = fv.step_newer(JUDGE)

## 7. Findings
Each pass rule of 8.2-8.4, read off the results; `results/summary.json` and `results/findings.md` under WORK.

In [ ]:
summary = fv.step_findings(outs)
print(open(fv.paths()['results'] / 'findings.md').read())

## Verdict

In [ ]:
for k, v in summary['verdict'].items():
    print(('holds   ' if v else 'FAILS   ') + k)
print(f"spent ${fv.spent():.2f} of ${fv.TOTAL_MAX_USD}")

## 8. Upgrades, from the tested results

`.docs/p4/upgrades/index.json` lists the upgrades the precision upgrader wrote, one directory each (`patch.diff`,
`test.sh` printing one JSON line `{"metric", "before", "after", "pass"}`, a README). In order, each is applied to a
scratch copy of the worktree with the upgrades already kept; it is kept only when its `test.sh` passes and every free
test passes there; one that no longer applies over the kept ones is skipped as a conflict. Then the free measures
run again on the kept set (the floor, silence and the regression queries, latency). With `UPGRADES_PAID`, the kept
set is re-measured with the paid steps too, within what is left of the cap. With `APPLY_UPGRADES`, the kept set is
one commit on this branch, with `.docs/p4/upgrades/results.md`; nothing is pushed.

In [ ]:
upgrades = fv.step_upgrades()
print(json.dumps({k: upgrades[k] for k in ('kept', 'committed')}, indent=1))